In [0]:
# ============================================================
# BLS DATA QUEST
# END-TO-END DATA QUALITY VALIDATION
# ============================================================
#
# Catalog:
#   bls_dataquest
#
# Schemas:
#   bronze
#   silver
#   gold
#   statistical
#
# This notebook validates the resulting data products.
# It does NOT modify any Lakeflow pipeline.
# ============================================================


from pyspark.sql import functions as F


# ============================================================
# 0. CONFIGURATION
# ============================================================

CATALOG = "bls_dataquest"

print("=" * 70)
print("BLS DATA QUEST - DATA QUALITY VALIDATION")
print("=" * 70)

print(f"Catalog: {CATALOG}")


# ============================================================
# 1. HELPER FUNCTIONS
# ============================================================

validation_results = []


def record_check(
    layer,
    check_name,
    passed,
    details=""
):

    validation_results.append(
        (
            layer,
            check_name,
            bool(passed),
            details
        )
    )


def print_check(
    layer,
    check_name,
    passed,
    details=""
):

    status = "PASS" if passed else "FAIL"

    print(
        f"[{status}] "
        f"{layer} | "
        f"{check_name}"
        + (
            f" | {details}"
            if details
            else ""
        )
    )


def run_check(
    layer,
    check_name,
    passed,
    details=""
):

    record_check(
        layer,
        check_name,
        passed,
        details
    )

    print_check(
        layer,
        check_name,
        passed,
        details
    )


# ============================================================
# 2. BRONZE TABLE DEFINITIONS
# ============================================================

bronze_tables = [
    f"{CATALOG}.bronze.pr_class",
    f"{CATALOG}.bronze.pr_data_current",
    f"{CATALOG}.bronze.pr_duration",
    f"{CATALOG}.bronze.pr_footnote",
    f"{CATALOG}.bronze.pr_measure",
    f"{CATALOG}.bronze.pr_period",
    f"{CATALOG}.bronze.pr_seasonal",
    f"{CATALOG}.bronze.pr_sector",
    f"{CATALOG}.bronze.pr_series",
    f"{CATALOG}.bronze.population"
]


# ============================================================
# 3. BRONZE VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("1. BRONZE VALIDATION")
print("=" * 70)


bronze_row_counts = []


for table_name in bronze_tables:

    try:

        row_count = (
            spark
            .read
            .table(table_name)
            .count()
        )

        bronze_row_counts.append(
            (
                table_name,
                row_count
            )
        )

        run_check(
            "BRONZE",
            f"{table_name} populated",
            row_count > 0,
            f"rows={row_count:,}"
        )

    except Exception as e:

        bronze_row_counts.append(
            (
                table_name,
                None
            )
        )

        run_check(
            "BRONZE",
            f"{table_name} accessible",
            False,
            str(e)
        )


# ============================================================
# 4. BRONZE - POPULATION
# ============================================================

try:

    bronze_population = spark.read.table(
        f"{CATALOG}.bronze.population"
    )

    population_count = bronze_population.count()

    run_check(
        "BRONZE",
        "Population contains records",
        population_count > 0,
        f"rows={population_count:,}"
    )


    population_null_years = (
        bronze_population
        .filter(
            F.col("year").isNull()
        )
        .count()
    )

    run_check(
        "BRONZE",
        "Population year is not NULL",
        population_null_years == 0,
        f"null_years={population_null_years}"
    )


except Exception as e:

    run_check(
        "BRONZE",
        "Population validation",
        False,
        str(e)
    )


# ============================================================
# 5. SILVER VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("2. SILVER VALIDATION")
print("=" * 70)


# ============================================================
# 5.1 PRODUCTIVITY OBSERVATIONS
# ============================================================

try:

    silver_observations = spark.read.table(
        f"{CATALOG}.silver.productivity_observations"
    )

    silver_count = silver_observations.count()

    run_check(
        "SILVER",
        "Productivity observations populated",
        silver_count > 0,
        f"rows={silver_count:,}"
    )


    # --------------------------------------------------------
    # NULL CHECKS
    # --------------------------------------------------------

    null_series_id = (
        silver_observations
        .filter(
            F.col("series_id").isNull()
        )
        .count()
    )

    null_year = (
        silver_observations
        .filter(
            F.col("year").isNull()
        )
        .count()
    )

    null_period = (
        silver_observations
        .filter(
            F.col("period").isNull()
        )
        .count()
    )

    null_value = (
        silver_observations
        .filter(
            F.col("value").isNull()
        )
        .count()
    )


    run_check(
        "SILVER",
        "series_id has no NULL values",
        null_series_id == 0,
        f"nulls={null_series_id}"
    )

    run_check(
        "SILVER",
        "year has no NULL values",
        null_year == 0,
        f"nulls={null_year}"
    )

    run_check(
        "SILVER",
        "period has no NULL values",
        null_period == 0,
        f"nulls={null_period}"
    )

    run_check(
        "SILVER",
        "value has no NULL values",
        null_value == 0,
        f"nulls={null_value}"
    )


    # --------------------------------------------------------
    # YEAR VALIDATION
    # --------------------------------------------------------

    invalid_years = (
        silver_observations
        .filter(
            (F.col("year") < 1900)
            | (F.col("year") > 2100)
        )
        .count()
    )

    run_check(
        "SILVER",
        "Year values are valid",
        invalid_years == 0,
        f"invalid_years={invalid_years}"
    )


    # --------------------------------------------------------
    # PERIOD VALIDATION
    # --------------------------------------------------------

    invalid_periods = (
        silver_observations
        .filter(
            ~F.col("period").isin(
                "Q01",
                "Q02",
                "Q03",
                "Q04"
            )
        )
        .count()
    )

    run_check(
        "SILVER",
        "BLS periods are valid",
        invalid_periods == 0,
        f"invalid_periods={invalid_periods}"
    )


    # --------------------------------------------------------
    # DUPLICATE VALIDATION
    # --------------------------------------------------------

    duplicate_observations = (
        silver_observations
        .groupBy(
            "series_id",
            "year",
            "period"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    run_check(
        "SILVER",
        "Observation grain is unique",
        duplicate_observations == 0,
        f"duplicate_groups={duplicate_observations}"
    )


except Exception as e:

    run_check(
        "SILVER",
        "Productivity observations accessible",
        False,
        str(e)
    )


# ============================================================
# 5.2 PRODUCTIVITY SERIES
# ============================================================

try:

    silver_series = spark.read.table(
        f"{CATALOG}.silver.productivity_series"
    )

    series_count = silver_series.count()

    run_check(
        "SILVER",
        "Productivity series populated",
        series_count > 0,
        f"rows={series_count:,}"
    )


    duplicate_series_ids = (
        silver_series
        .groupBy("series_id")
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    run_check(
        "SILVER",
        "Series IDs are unique",
        duplicate_series_ids == 0,
        f"duplicate_series_ids={duplicate_series_ids}"
    )


    null_sector_name = (
        silver_series
        .filter(
            F.col("sector_name").isNull()
        )
        .count()
    )

    null_measure_text = (
        silver_series
        .filter(
            F.col("measure_text").isNull()
        )
        .count()
    )


    run_check(
        "SILVER",
        "Sector metadata populated",
        null_sector_name == 0,
        f"null_sector_name={null_sector_name}"
    )

    run_check(
        "SILVER",
        "Measure metadata populated",
        null_measure_text == 0,
        f"null_measure_text={null_measure_text}"
    )


except Exception as e:

    run_check(
        "SILVER",
        "Productivity series validation",
        False,
        str(e)
    )


# ============================================================
# 5.3 POPULATION CLEAN
# ============================================================

try:

    silver_population = spark.read.table(
        f"{CATALOG}.silver.population_clean"
    )

    population_count = silver_population.count()

    run_check(
        "SILVER",
        "Clean population populated",
        population_count > 0,
        f"rows={population_count:,}"
    )


    duplicate_population_years = (
        silver_population
        .groupBy("year")
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    run_check(
        "SILVER",
        "Population has one record per year",
        duplicate_population_years == 0,
        f"duplicate_years={duplicate_population_years}"
    )


    invalid_population = (
        silver_population
        .filter(
            F.col("population") <= 0
        )
        .count()
    )

    run_check(
        "SILVER",
        "Population values are positive",
        invalid_population == 0,
        f"invalid_population={invalid_population}"
    )


except Exception as e:

    run_check(
        "SILVER",
        "Population validation",
        False,
        str(e)
    )


# ============================================================
# 5.4 ENRICHED PRODUCTIVITY
# ============================================================

try:

    silver_enriched = spark.read.table(
        f"{CATALOG}.silver.productivity_enriched"
    )

    enriched_count = silver_enriched.count()

    run_check(
        "SILVER",
        "Enriched productivity populated",
        enriched_count > 0,
        f"rows={enriched_count:,}"
    )


    enriched_duplicates = (
        silver_enriched
        .groupBy(
            "series_id",
            "year",
            "period"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    run_check(
        "SILVER",
        "Enriched observation grain remains unique",
        enriched_duplicates == 0,
        f"duplicate_groups={enriched_duplicates}"
    )


except Exception as e:

    run_check(
        "SILVER",
        "Enriched productivity validation",
        False,
        str(e)
    )


# ============================================================
# 6. GOLD VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("3. GOLD VALIDATION")
print("=" * 70)


# ============================================================
# 6.1 DIMENSION SERIES
# ============================================================

try:

    dim_series = spark.read.table(
        f"{CATALOG}.gold.dim_series"
    )

    dim_count = dim_series.count()

    run_check(
        "GOLD",
        "dim_series populated",
        dim_count > 0,
        f"rows={dim_count:,}"
    )


    duplicate_dim_series = (
        dim_series
        .groupBy("series_id")
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    run_check(
        "GOLD",
        "dim_series has one row per series",
        duplicate_dim_series == 0,
        f"duplicate_series={duplicate_dim_series}"
    )


    null_series_labels = (
        dim_series
        .filter(
            F.col("series_label").isNull()
            |
            (
                F.trim(
                    F.col("series_label")
                ) == ""
            )
        )
        .count()
    )

    run_check(
        "GOLD",
        "Series labels populated",
        null_series_labels == 0,
        f"missing_labels={null_series_labels}"
    )


except Exception as e:

    run_check(
        "GOLD",
        "dim_series validation",
        False,
        str(e)
    )


# ============================================================
# 6.2 FACT PRODUCTIVITY
# ============================================================

try:

    fact_productivity = spark.read.table(
        f"{CATALOG}.gold.fact_productivity"
    )

    fact_count = fact_productivity.count()

    run_check(
        "GOLD",
        "fact_productivity populated",
        fact_count > 0,
        f"rows={fact_count:,}"
    )


    invalid_quarters = (
    fact
    .filter(
        (F.col("period_type") == "QUARTERLY")
        & (
            ~F.col("quarter").isin([1, 2, 3, 4])
            | F.col("quarter").isNull()
        )
    )
    .count()
    )

    run_check(
        "GOLD",
        "Quarter values are valid",
        invalid_quarters == 0,
        f"invalid_quarters={invalid_quarters}"
    )


    null_observation_dates = (
        fact_productivity
        .filter(
            F.col("observation_date").isNull()
        )
        .count()
    )

    run_check(
        "GOLD",
        "Observation dates are populated",
        null_observation_dates == 0,
        f"null_dates={null_observation_dates}"
    )


    fact_duplicates = (
        fact_productivity
        .groupBy(
            "series_id",
            "year",
            "period"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    run_check(
        "GOLD",
        "Fact observation grain is unique",
        fact_duplicates == 0,
        f"duplicate_groups={fact_duplicates}"
    )


except Exception as e:

    run_check(
        "GOLD",
        "fact_productivity validation",
        False,
        str(e)
    )


# ============================================================
# 6.3 ANNUAL METRICS
# ============================================================

try:

    annual_metrics = spark.read.table(
        f"{CATALOG}.gold.series_annual_metrics"
    )

    annual_count = annual_metrics.count()

    run_check(
        "GOLD",
        "Annual metrics populated",
        annual_count > 0,
        f"rows={annual_count:,}"
    )


    annual_duplicates = (
        annual_metrics
        .groupBy(
            "series_id",
            "year"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    run_check(
        "GOLD",
        "Annual metric grain is unique",
        annual_duplicates == 0,
        f"duplicate_groups={annual_duplicates}"
    )


    best_year_count = (
        annual_metrics
        .filter(
            F.col("is_best_year") == True
        )
        .groupBy("series_id")
        .count()
        .filter(
            F.col("count") != 1
        )
        .count()
    )

    run_check(
        "GOLD",
        "Each series has exactly one best year",
        best_year_count == 0,
        f"invalid_series={best_year_count}"
    )


except Exception as e:

    run_check(
        "GOLD",
        "Annual metrics validation",
        False,
        str(e)
    )


# ============================================================
# 6.4 QUARTERLY METRICS
# ============================================================

try:

    quarterly_metrics = spark.read.table(
        f"{CATALOG}.gold.series_quarterly_metrics"
    )

    quarterly_count = quarterly_metrics.count()

    run_check(
        "GOLD",
        "Quarterly metrics populated",
        quarterly_count > 0,
        f"rows={quarterly_count:,}"
    )


    quarterly_duplicates = (
        quarterly_metrics
        .groupBy(
            "series_id",
            "year",
            "period"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    run_check(
        "GOLD",
        "Quarterly metric grain is unique",
        quarterly_duplicates == 0,
        f"duplicate_groups={quarterly_duplicates}"
    )


except Exception as e:

    run_check(
        "GOLD",
        "Quarterly metrics validation",
        False,
        str(e)
    )


# ============================================================
# 6.5 POPULATION CONTEXT
# ============================================================

try:

    population_context = spark.read.table(
        f"{CATALOG}.gold.population_context"
    )

    population_context_count = (
        population_context.count()
    )

    run_check(
        "GOLD",
        "Population context populated",
        population_context_count > 0,
        f"rows={population_context_count:,}"
    )


    duplicate_population_context = (
        population_context
        .groupBy("year")
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    run_check(
        "GOLD",
        "Population context has one row per year",
        duplicate_population_context == 0,
        f"duplicate_years={duplicate_population_context}"
    )


except Exception as e:

    run_check(
        "GOLD",
        "Population context validation",
        False,
        str(e)
    )


# ============================================================
# 6.6 SERIES MOVEMENT
# ============================================================

try:

    series_movement = spark.read.table(
        f"{CATALOG}.gold.series_movement"
    )

    movement_count = series_movement.count()

    run_check(
        "GOLD",
        "Series movement populated",
        movement_count > 0,
        f"rows={movement_count:,}"
    )


    invalid_directions = (
        series_movement
        .filter(
            ~F.col("movement_direction").isin(
                "INCREASE",
                "DECREASE",
                "NO_CHANGE"
            )
        )
        .count()
    )

    run_check(
        "GOLD",
        "Movement directions are valid",
        invalid_directions == 0,
        f"invalid_directions={invalid_directions}"
    )


    invalid_magnitudes = (
        series_movement
        .filter(
            ~F.col("movement_magnitude").isin(
                "UNKNOWN",
                "STABLE",
                "MODERATE",
                "LARGE"
            )
        )
        .count()
    )

    run_check(
        "GOLD",
        "Movement magnitudes are valid",
        invalid_magnitudes == 0,
        f"invalid_magnitudes={invalid_magnitudes}"
    )


except Exception as e:

    run_check(
        "GOLD",
        "Series movement validation",
        False,
        str(e)
    )


# ============================================================
# 7. STATISTICAL / ANOMALY VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("4. STATISTICAL / ANOMALY VALIDATION")
print("=" * 70)


# ============================================================
# 7.1 ANOMALY SCORES
# ============================================================

try:

    anomaly_scores = spark.read.table(
        f"{CATALOG}.statistical.anomaly_scores"
    )

    anomaly_score_count = anomaly_scores.count()

    run_check(
        "STATISTICAL",
        "Anomaly scores populated",
        anomaly_score_count > 0,
        f"rows={anomaly_score_count:,}"
    )


    invalid_severity = (
        anomaly_scores
        .filter(
            ~F.col("severity").isin(
                "NORMAL",
                "MEDIUM",
                "HIGH",
                "CRITICAL",
                "INSUFFICIENT_HISTORY"
            )
        )
        .count()
    )

    run_check(
        "STATISTICAL",
        "Anomaly severity values are valid",
        invalid_severity == 0,
        f"invalid_severity={invalid_severity}"
    )


    invalid_baseline_status = (
        anomaly_scores
        .filter(
            (
                F.col("baseline_ready") == True
            )
            &
            (
                F.col("baseline_observation_count") < 4
            )
        )
        .count()
    )

    run_check(
        "STATISTICAL",
        "Baseline readiness is consistent",
        invalid_baseline_status == 0,
        f"invalid_rows={invalid_baseline_status}"
    )


    negative_scores = (
        anomaly_scores
        .filter(
            F.col("anomaly_score") < 0
        )
        .count()
    )

    run_check(
        "STATISTICAL",
        "Anomaly scores are non-negative",
        negative_scores == 0,
        f"negative_scores={negative_scores}"
    )


except Exception as e:

    run_check(
        "STATISTICAL",
        "Anomaly scores validation",
        False,
        str(e)
    )


# ============================================================
# 7.2 ANOMALY EVENTS
# ============================================================

try:

    anomaly_events = spark.read.table(
        f"{CATALOG}.statistical.anomaly_events"
    )

    anomaly_event_count = anomaly_events.count()

    print(
        f"\nDetected anomaly events: "
        f"{anomaly_event_count:,}"
    )


    invalid_event_severity = (
        anomaly_events
        .filter(
            ~F.col("severity").isin(
                "MEDIUM",
                "HIGH",
                "CRITICAL"
            )
        )
        .count()
    )

    run_check(
        "STATISTICAL",
        "Anomaly event severity is valid",
        invalid_event_severity == 0,
        f"invalid_events={invalid_event_severity}"
    )


    missing_reasons = (
        anomaly_events
        .filter(
            F.col("anomaly_reason").isNull()
            |
            (
                F.trim(
                    F.col("anomaly_reason")
                ) == ""
            )
        )
        .count()
    )

    run_check(
        "STATISTICAL",
        "All anomaly events have reasons",
        missing_reasons == 0,
        f"missing_reasons={missing_reasons}"
    )


    duplicate_events = (
        anomaly_events
        .groupBy(
            "series_id",
            "year",
            "period"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    run_check(
        "STATISTICAL",
        "Anomaly event grain is unique",
        duplicate_events == 0,
        f"duplicate_events={duplicate_events}"
    )


except Exception as e:

    run_check(
        "STATISTICAL",
        "Anomaly events validation",
        False,
        str(e)
    )


# ============================================================
# 8. CROSS-LAYER CONSISTENCY
# ============================================================

print("\n")
print("=" * 70)
print("5. CROSS-LAYER CONSISTENCY")
print("=" * 70)


# ============================================================
# SILVER → GOLD
# ============================================================

try:

    silver_keys = (
        silver_observations
        .select(
            "series_id",
            "year",
            "period"
        )
        .distinct()
    )

    gold_keys = (
        fact_productivity
        .select(
            "series_id",
            "year",
            "period"
        )
        .distinct()
    )


    missing_in_gold = (
        silver_keys
        .join(
            gold_keys,
            [
                "series_id",
                "year",
                "period"
            ],
            "left_anti"
        )
        .count()
    )


    run_check(
        "CROSS-LAYER",
        "Silver observations exist in Gold",
        missing_in_gold == 0,
        f"missing_in_gold={missing_in_gold}"
    )


except Exception as e:

    run_check(
        "CROSS-LAYER",
        "Silver → Gold consistency",
        False,
        str(e)
    )


# ============================================================
# GOLD → STATISTICAL
# ============================================================

try:

    gold_keys = (
        fact_productivity
        .select(
            "series_id",
            "year",
            "period"
        )
        .distinct()
    )

    statistical_keys = (
        anomaly_scores
        .select(
            "series_id",
            "year",
            "period"
        )
        .distinct()
    )


    missing_in_statistical = (
        gold_keys
        .join(
            statistical_keys,
            [
                "series_id",
                "year",
                "period"
            ],
            "left_anti"
        )
        .count()
    )


    # Informational only.
    #
    # Early observations can legitimately have insufficient
    # historical data for anomaly calculations.

    print(
        "Gold observations without statistical records: "
        f"{missing_in_statistical:,}"
    )


except Exception as e:

    run_check(
        "CROSS-LAYER",
        "Gold → Statistical coverage",
        False,
        str(e)
    )


# ============================================================
# 9. DATA VOLUME SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("6. DATA VOLUME SUMMARY")
print("=" * 70)


volume_summary = []


# ------------------------------------------------------------
# Bronze
# ------------------------------------------------------------

for table_name, row_count in bronze_row_counts:

    volume_summary.append(
        (
            "BRONZE",
            table_name,
            row_count
        )
    )


# ------------------------------------------------------------
# Silver
# ------------------------------------------------------------

try:

    volume_summary.extend([
        (
            "SILVER",
            f"{CATALOG}.silver.productivity_observations",
            silver_observations.count()
        ),
        (
            "SILVER",
            f"{CATALOG}.silver.productivity_series",
            silver_series.count()
        ),
        (
            "SILVER",
            f"{CATALOG}.silver.population_clean",
            silver_population.count()
        ),
        (
            "SILVER",
            f"{CATALOG}.silver.productivity_enriched",
            silver_enriched.count()
        )
    ])

except Exception:
    pass


# ------------------------------------------------------------
# Gold
# ------------------------------------------------------------

try:

    volume_summary.extend([
        (
            "GOLD",
            f"{CATALOG}.gold.dim_series",
            dim_series.count()
        ),
        (
            "GOLD",
            f"{CATALOG}.gold.fact_productivity",
            fact_productivity.count()
        ),
        (
            "GOLD",
            f"{CATALOG}.gold.series_annual_metrics",
            annual_metrics.count()
        ),
        (
            "GOLD",
            f"{CATALOG}.gold.series_quarterly_metrics",
            quarterly_metrics.count()
        ),
        (
            "GOLD",
            f"{CATALOG}.gold.population_context",
            population_context.count()
        ),
        (
            "GOLD",
            f"{CATALOG}.gold.series_movement",
            series_movement.count()
        )
    ])

except Exception:
    pass


# ------------------------------------------------------------
# Statistical
# ------------------------------------------------------------

try:

    volume_summary.extend([
        (
            "STATISTICAL",
            f"{CATALOG}.statistical.series_baselines",
            spark.read.table(
                f"{CATALOG}.statistical.series_baselines"
            ).count()
        ),
        (
            "STATISTICAL",
            f"{CATALOG}.statistical.anomaly_scores",
            anomaly_scores.count()
        ),
        (
            "STATISTICAL",
            f"{CATALOG}.statistical.anomaly_events",
            anomaly_events.count()
        )
    ])

except Exception:
    pass


volume_df = spark.createDataFrame(
    volume_summary,
    [
        "layer",
        "table_name",
        "row_count"
    ]
)


display(
    volume_df
    .orderBy(
        "layer",
        "table_name"
    )
)


# ============================================================
# 10. FINAL QUALITY REPORT
# ============================================================

print("\n")
print("=" * 70)
print("7. FINAL DATA QUALITY REPORT")
print("=" * 70)


quality_df = spark.createDataFrame(
    validation_results,
    [
        "layer",
        "check_name",
        "passed",
        "details"
    ]
)


display(
    quality_df
    .orderBy(
        "layer",
        "check_name"
    )
)


# ============================================================
# 11. FINAL PASS / FAIL SUMMARY
# ============================================================

total_checks = quality_df.count()

failed_checks = (
    quality_df
    .filter(
        ~F.col("passed")
    )
    .count()
)

passed_checks = (
    total_checks
    - failed_checks
)


print("\n")
print("=" * 70)
print("FINAL RESULT")
print("=" * 70)

print(
    f"Catalog      : {CATALOG}"
)

print(
    f"Total checks : {total_checks}"
)

print(
    f"Passed       : {passed_checks}"
)

print(
    f"Failed       : {failed_checks}"
)

print(
    f"Pass rate    : "
    f"{(passed_checks / total_checks * 100):.2f}%"
    if total_checks > 0
    else "N/A"
)

print("=" * 70)


# ============================================================
# 12. FAIL NOTEBOOK IF ANY CHECK FAILED
# ============================================================

if failed_checks > 0:

    print(
        "\nDATA QUALITY VALIDATION FAILED."
    )

    display(
        quality_df
        .filter(
            ~F.col("passed")
        )
    )

    raise ValueError(
        f"Data quality validation failed: "
        f"{failed_checks} check(s) failed."
    )


else:

    print(
        "\nDATA QUALITY VALIDATION PASSED."
    )

    print(
        "All validation checks completed successfully."
    )

BLS DATA QUEST - DATA QUALITY VALIDATION
Catalog: bls_dataquest


1. BRONZE VALIDATION
[PASS] BRONZE | bls_dataquest.bronze.pr_class populated | rows=2
[PASS] BRONZE | bls_dataquest.bronze.pr_data_current populated | rows=38,469
[PASS] BRONZE | bls_dataquest.bronze.pr_duration populated | rows=3
[PASS] BRONZE | bls_dataquest.bronze.pr_footnote populated | rows=1
[PASS] BRONZE | bls_dataquest.bronze.pr_measure populated | rows=22
[PASS] BRONZE | bls_dataquest.bronze.pr_period populated | rows=5
[PASS] BRONZE | bls_dataquest.bronze.pr_seasonal populated | rows=2
[PASS] BRONZE | bls_dataquest.bronze.pr_sector populated | rows=6
[PASS] BRONZE | bls_dataquest.bronze.pr_series populated | rows=282
[PASS] BRONZE | bls_dataquest.bronze.population populated | rows=12
[PASS] BRONZE | Population contains records | rows=12
[PASS] BRONZE | Population year is not NULL | null_years=0


2. SILVER VALIDATION
[PASS] SILVER | Productivity observations populated | rows=38,469
[PASS] SILVER | series_id has

layer,table_name,row_count
BRONZE,bls_dataquest.bronze.population,12
BRONZE,bls_dataquest.bronze.pr_class,2
BRONZE,bls_dataquest.bronze.pr_data_current,38469
BRONZE,bls_dataquest.bronze.pr_duration,3
BRONZE,bls_dataquest.bronze.pr_footnote,1
BRONZE,bls_dataquest.bronze.pr_measure,22
BRONZE,bls_dataquest.bronze.pr_period,5
BRONZE,bls_dataquest.bronze.pr_seasonal,2
BRONZE,bls_dataquest.bronze.pr_sector,6
BRONZE,bls_dataquest.bronze.pr_series,282




7. FINAL DATA QUALITY REPORT


layer,check_name,passed,details
BRONZE,Population contains records,true,rows=12
BRONZE,Population year is not NULL,true,null_years=0
BRONZE,bls_dataquest.bronze.population populated,true,rows=12
BRONZE,bls_dataquest.bronze.pr_class populated,true,rows=2
BRONZE,bls_dataquest.bronze.pr_data_current populated,true,"rows=38,469"
BRONZE,bls_dataquest.bronze.pr_duration populated,true,rows=3
BRONZE,bls_dataquest.bronze.pr_footnote populated,true,rows=1
BRONZE,bls_dataquest.bronze.pr_measure populated,true,rows=22
BRONZE,bls_dataquest.bronze.pr_period populated,true,rows=5
BRONZE,bls_dataquest.bronze.pr_seasonal populated,true,rows=2




FINAL RESULT
Catalog      : bls_dataquest
Total checks : 54
Passed       : 51
Failed       : 3
Pass rate    : 94.44%

DATA QUALITY VALIDATION FAILED.


layer,check_name,passed,details
SILVER,BLS periods are valid,false,invalid_periods=8607
GOLD,Quarter values are valid,false,invalid_quarters=8607
GOLD,Observation dates are populated,false,null_dates=8607


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-8867847089520518>, line 1582
   1571     print(
   1572         "\nDATA QUALITY VALIDATION FAILED."
   1573     )
   1575     display(
   1576         quality_df
   1577         .filter(
   1578             ~F.col("passed")
   1579         )
   1580     )
-> 1582     raise ValueError(
   1583         f"Data quality validation failed: "
   1584         f"{failed_checks} check(s) failed."
   1585     )
   1588 else:
   1590     print(
   1591         "\nDATA QUALITY VALIDATION PASSED."
   1592     )

ValueError: Data quality validation failed: 3 check(s) failed.